## 🧪 Classification Testing Notebook Overview

This notebook provides a complete, plug-and-play interface for evaluating our trained **cervical cancer classification model** on a set of labeled testing images. It is designed so that anyone can load the classifier, run predictions, compute accuracy/metrics, and visually inspect results — without needing to understand the underlying training pipeline.

### What this notebook does

- **Loads the trained classifier** saved in the `models/` folder  
  - `best_classifier_placeholder.pt` — placeholder name; replace with the actual model filename

- **Loads the testing images** and their corresponding labels from:  
  - `testing_images/` — contains all testing images  
  - `test_tags.csv` — contains ground-truth labels (`Cancer status`: 0 = Normal, 1 = Pre-cancerous, 2 = Cancerous)

- **Runs inference** on all testing images and computes:  
  - Overall accuracy  
  - Per-class precision, recall, and F1 score  
  - Confusion matrix (raw + normalized)

- Provides **utility functions** to:
  - Visualize a random sample of predictions  
  - Show predicted vs. true label and probability distribution per image  
  - Debug misclassifications and inspect model behavior

### Why this notebook is useful

- Provides a standardized evaluation workflow for all classification models.  
- Ensures consistent comparison across different backbones or training strategies.  
- Facilitates quick visual QC of model predictions and confidence scores.  
- Helps identify systematic errors (e.g., confusion between pre-cancerous vs. cancerous).

### Folder structure expected

```
for_testing/
│
├── testing_classification.ipynb # ← this notebook
│
├── testing_images/ # testing images + labels
│ ├── via_img_01.jpg
│ ├── via_img_02.png
│ ├── ...
│ └── test_tags.csv # contains "Cancer status" labels
│
└── models/
└── best_classifier_placeholder.pt
```


Use this notebook as a unified, repeatable way to test any new classification models we produce.


In [ ]:
import os
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

# -------------------
# Paths (edit if needed)
# -------------------
ROOT_DIR = Path(".")  # notebook inside for_testing/
TEST_ROOT = ROOT_DIR

IMAGE_DIR = TEST_ROOT / "testing_images"
CSV_PATH  = IMAGE_DIR / "test_tags.csv"
MODEL_DIR = TEST_ROOT / "models"

# 🔁 placeholder model name – change this when you have the real file
CLASSIFIER_WEIGHTS = MODEL_DIR / "best_classifier_placeholder.pt"

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Class mapping: 0-Normal, 1-Pre-cancerous, 2-Cancerous
CLASS_NAMES = ["Normal", "Pre-cancerous", "Cancerous"]
NUM_CLASSES = len(CLASS_NAMES)

print("Images path:", IMAGE_DIR)
print("CSV path   :", CSV_PATH)
print("Model path :", CLASSIFIER_WEIGHTS)


In [ ]:
# Read CSV
df = pd.read_csv(CSV_PATH)

print("Columns in CSV:", list(df.columns))

# Try to infer label column name
label_col_candidates = ["Cancer status", "Cancer stat", "cancer_status", "cancer_stat"]
label_col = None
for c in label_col_candidates:
    if c in df.columns:
        label_col = c
        break

if label_col is None:
    raise ValueError(
        f"Could not find cancer-status column in CSV. "
        f"Tried: {label_col_candidates}"
    )

print(f"Using label column: '{label_col}'")

# Helper: find image path for a given stem
IMG_EXTS = [".jpg", ".jpeg", ".png"]

def find_image_path(stem: str):
    for ext in IMG_EXTS:
        candidate = IMAGE_DIR / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None

records = []

for _, row in df.iterrows():
    stem = str(row["image"])
    label_int = int(row[label_col])

    img_path = find_image_path(stem)
    if img_path is None:
        print(f"⚠️ Image file not found for stem '{stem}' – skipping")
        continue

    records.append(
        {
            "stem": stem,
            "path": img_path,
            "label": label_int,
        }
    )

print(f"\nResolved {len(records)} images from CSV.")
if len(records) == 0:
    raise RuntimeError("No images resolved – check filenames / extensions.")

# Convert to DataFrame for convenience
records_df = pd.DataFrame(records)
records_df.head()


In [ ]:
# Standard ImageNet-like transforms (adjust if your model used something else)
IMG_SIZE = 224

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


class TestClassificationDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img_path = rec["path"]
        label = rec["label"]

        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)

        return img, label, str(img_path)


test_dataset = TestClassificationDataset(records, transform=test_transform)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Test dataset size: {len(test_dataset)} images")


In [ ]:
def build_placeholder_model(num_classes: int = NUM_CLASSES):
    """
    Placeholder architecture in case you only saved state_dict.
    Replace this with your actual model definition if needed.
    """
    # Example: simple ResNet18 classification head
    from torchvision.models import resnet18

    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def load_classifier(weights_path: Path, num_classes: int = NUM_CLASSES):
    """
    Try to load the classifier. By default we assume the entire model was saved.

    If that fails, we fall back to building a placeholder architecture and
    loading the weights as a state_dict.
    """
    if not weights_path.exists():
        raise FileNotFoundError(f"Classifier weights not found at: {weights_path}")

    try:
        # Case 1: file contains a full model object
        model = torch.load(weights_path, map_location=device)
        print("Loaded full model object from .pt file.")
    except Exception as e:
        print("Could not load full model directly:", e)
        print("Trying to load as state_dict into placeholder architecture...")
        model = build_placeholder_model(num_classes=num_classes)
        state = torch.load(weights_path, map_location=device)
        model.load_state_dict(state, strict=False)

    model.to(device)
    model.eval()
    return model


classifier_model = load_classifier(CLASSIFIER_WEIGHTS, num_classes=NUM_CLASSES)
print("Model loaded and ready for evaluation.")


In [ ]:
all_labels = []
all_preds  = []
all_probs  = []
all_paths  = []

softmax = nn.Softmax(dim=1)

classifier_model.eval()
with torch.no_grad():
    for imgs, labels, paths in test_loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        logits = classifier_model(imgs)
        probs  = softmax(logits)
        preds  = probs.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy())
        all_paths.extend(paths)

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

print(f"Evaluated {len(all_labels)} images.")


In [ ]:
# Basic accuracy
accuracy = (all_labels == all_preds).mean()
print(f"Overall accuracy: {accuracy:.3f}")

# Detailed classification report
print("\nClassification report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2])
cm


In [ ]:
def plot_confusion_matrix(cm, class_names, normalize=False):
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1, keepdims=True).clip(min=1e-9)

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.figure.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True label",
        xlabel="Predicted label",
        title="Confusion matrix" + (" (normalized)" if normalize else ""),
    )

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    fmt = ".2f" if normalize else "d"
    thresh = cm.max() / 2.0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                format(cm[i, j], fmt),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black",
            )

    fig.tight_layout()
    plt.show()


plot_confusion_matrix(cm, CLASS_NAMES, normalize=False)
plot_confusion_matrix(cm, CLASS_NAMES, normalize=True)


In [ ]:
def show_predictions(n=6):
    n = min(n, len(all_paths))
    indices = np.random.choice(len(all_paths), size=n, replace=False)

    plt.figure(figsize=(12, 2.5 * n))

    for i, idx in enumerate(indices, start=1):
        img_path = all_paths[idx]
        true_lbl = all_labels[idx]
        pred_lbl = all_preds[idx]
        probs    = all_probs[idx]

        img = Image.open(img_path).convert("RGB")

        plt.subplot(n, 2, 2 * (i - 1) + 1)
        plt.imshow(img)
        plt.axis("off")

        title = f"{Path(img_path).name}\nTrue: {CLASS_NAMES[true_lbl]} | Pred: {CLASS_NAMES[pred_lbl]}"
        color = "green" if true_lbl == pred_lbl else "red"
        plt.title(title, color=color)

        # Plot probability bar chart
        plt.subplot(n, 2, 2 * (i - 1) + 2)
        plt.bar(range(NUM_CLASSES), probs)
        plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45)
        plt.ylim(0, 1)
        plt.ylabel("Probability")
        plt.title("Class probabilities")

    plt.tight_layout()
    plt.show()


show_predictions(n=6)
